# Qwen3.5-9B — klasifikasi hate speech

**Alur utama: SFT LoRA BF16**, dengan baseline zero-shot opsional. Thinking dimatikan dan prompt tanpa contoh berlabel.
Pakai satu H200 di Modal. Upload tiga CSV satu bahasa dan file prompt bahasa tersebut ke folder kerja, pilih `LANGUAGE`, lalu jalankan berurutan.
Contoh: `en-train.csv`, `en-val.csv`, `en-test.csv`, dan `EN_PROMPT.md`. Untuk Italian gunakan `IT_PROMPT.md`; Dutch `NL_PROMPT.md`. Folder split dan prompt repository juga dikenali.

Train untuk SFT; validation untuk checkpoint terbaik; test untuk evaluasi akhir.
Download ZIP sebelum sesi cloud berakhir. Untuk bahasa berikutnya, ubah `LANGUAGE` dan ulangi mulai bagian 2.
Training model 9B pada H200 belum dijalankan dalam penyusunan notebook ini.

## 1. Instalasi

Jalankan sekali, **restart kernel**, lalu lanjut bagian 2. PyTorch CUDA bawaan Modal dipertahankan.
Kernel cepat opsional memerlukan image dengan toolkit CUDA/compiler yang sesuai; default memakai fallback PyTorch.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'transformers==5.20.2', 'peft==0.21.2', 'accelerate==1.15.0',
    'datasets>=4,<6', 'pandas>=2.2,<3', 'scikit-learn>=1.6,<2', 'matplotlib>=3.9,<4'])
INSTALL_FAST_KERNELS = False
if INSTALL_FAST_KERNELS:
    import os
    os.environ.setdefault('MAX_JOBS', '4')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-build-isolation',
                           'flash-linear-attention==0.5.2', 'causal-conv1d==1.7.0'])
print('Restart kernel, lalu lanjut bagian 2.')


## 2. Bahasa dan pengaturan

Yang perlu diubah biasanya hanya `LANGUAGE`. `RUN_ZERO_SHOT_BASELINE` mengatur baseline; SFT tetap dijalankan.
Prompt otomatis dibaca dari `<LANGUAGE>_PROMPT.md` (nama bahasa huruf besar), di folder kerja atau folder `prompt` repository.
Label jawaban mengikuti file prompt: `0 = no`, `1 = yes_explicit`, `2 = yes_implicit`.
`INPUT_MODE='with_video_context'` menambahkan judul/deskripsi sebagai eksperimen terpisah.

In [ ]:
LANGUAGE = 'en'  # 'en', 'it', atau 'nl'; satu bahasa per run.
RUN_ZERO_SHOT_BASELINE = True
INPUT_MODE = 'comment_only'  # atau 'with_video_context'
MAX_LENGTH = 1024
BATCH_SIZE = 4
GRAD_ACCUM = 4  # Effective batch = 16.
EVAL_BATCH_SIZE = 4  # Tiga kandidat per komentar.
EPOCHS = 5
LEARNING_RATE = 1e-4
SEED = 42
MODEL_NAME = 'Qwen/Qwen3.5-9B'
LABELS = ['no', 'yes_implicit', 'yes_explicit']
# Label dataset/laporan tetap asli; jawaban model mengikuti angka dalam file prompt.
LABEL_TO_ANSWER = {'no': '0', 'yes_explicit': '1', 'yes_implicit': '2'}
LANGUAGES = {'en': 'English', 'it': 'Italian', 'nl': 'Dutch'}

import os
os.environ.update(TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')

import gc
import hashlib
import json
import math
import re
import unicodedata
from contextlib import nullcontext
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
import peft
from datasets import Dataset
from huggingface_hub import HfApi
from IPython.display import FileLink, display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3_5ForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed

assert LANGUAGE in LANGUAGES and INPUT_MODE in {'comment_only', 'with_video_context'}
assert MAX_LENGTH >= 256 and min(BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE) > 0
if transformers.__version__ != '5.20.2' or peft.__version__ != '0.21.2':
    raise RuntimeError('Jalankan instalasi lalu restart kernel.')
if not torch.cuda.is_available() or torch.cuda.device_count() != 1 or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Pilih satu GPU CUDA yang mendukung BF16, misalnya H200.')
set_seed(SEED)
OUT = Path('results') / f"qwen35_{LANGUAGE}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
OUT.mkdir(parents=True, exist_ok=False)
print('GPU:', torch.cuda.get_device_name(0), '| Bahasa:', LANGUAGE, '| Hasil:', OUT)

PROMPT_NAME = f'{LANGUAGE.upper()}_PROMPT.md'
prompt_candidates = [Path('.') / PROMPT_NAME, Path('prompt') / PROMPT_NAME,
                     Path('Analisis/prompt') / PROMPT_NAME, Path('../prompt') / PROMPT_NAME]
PROMPT_PATH = next((path for path in prompt_candidates if path.is_file()), None)
if PROMPT_PATH is None:
    raise FileNotFoundError(
        f'Upload {PROMPT_NAME} bersama tiga CSV ke {Path.cwd()}, '
        'atau letakkan di folder prompt repository. Periksa LANGUAGE.')
PROMPT_BYTES = PROMPT_PATH.read_bytes()
SYSTEM_PROMPT = PROMPT_BYTES.decode('utf-8-sig').strip()
if not SYSTEM_PROMPT:
    raise ValueError(f'File prompt kosong: {PROMPT_PATH.resolve()}')
PROMPT_SHA256 = hashlib.sha256(PROMPT_BYTES).hexdigest()
print('Prompt:', PROMPT_PATH.resolve(), '| SHA256:', PROMPT_SHA256)



## 3. Baca dan periksa tiga CSV

Split asli dipakai langsung. Audit memeriksa kolom, isi, kelas, serta overlap ID, komentar, dan konteks video.

In [ ]:
names = [f'{LANGUAGE}-{split}.csv' for split in ('train', 'val', 'test')]
candidates = [Path('.'), Path('data'), Path(f'{LANGUAGE}-splitting'),
              Path('data') / f'{LANGUAGE}-splitting', Path('Analisis/data') / f'{LANGUAGE}-splitting',
              Path('../data') / f'{LANGUAGE}-splitting']
for DATA_DIR in candidates:
    present = [(DATA_DIR / name).is_file() for name in names]
    if any(present):
        if not all(present):
            raise FileNotFoundError(f'Lengkapi {names} dalam folder {DATA_DIR.resolve()}.')
        break
else:
    raise FileNotFoundError(f'Upload {names} ke {Path.cwd()}. Periksa LANGUAGE.')

frames, source_hashes, audit_keys = {}, {}, {}
normalize = lambda text: re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(text)).casefold()).strip()
for split, name in zip(('train', 'val', 'test'), names):
    frame = pd.read_csv(DATA_DIR / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    required = {'StereoQueerEval_id', 'yt_comment', 'hate_speech'}
    if INPUT_MODE == 'with_video_context':
        required |= {'yt_title', 'yt_description'}
    if not required <= set(frame.columns) or frame.empty:
        raise ValueError(f'{name}: kolom wajib hilang atau split kosong.')
    if frame[['StereoQueerEval_id', 'yt_comment']].apply(lambda column: column.str.strip().eq('')).any().any():
        raise ValueError(f'{name}: ID/komentar kosong.')
    if frame['StereoQueerEval_id'].duplicated().any() or set(frame['hate_speech']) != set(LABELS):
        raise ValueError(f'{name}: ID duplikat atau kelas tidak sesuai.')
    audit_keys[split] = {'id': set(frame['StereoQueerEval_id']), 'comment': set(frame['yt_comment'].map(normalize))}
    if {'yt_title', 'yt_description'} <= set(frame.columns):
        contexts = zip(frame['yt_title'].map(normalize), frame['yt_description'].map(normalize))
        audit_keys[split]['video'] = {pair for pair in contexts if any(pair)}
    frames[split] = frame
    source_hashes[name] = hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
for left, right in combinations(frames, 2):
    for key in audit_keys[left].keys() & audit_keys[right].keys():
        if audit_keys[left][key] & audit_keys[right][key]:
            raise ValueError(f'Overlap {key}: {left} dan {right}. Periksa split asli.')
print('Folder CSV:', DATA_DIR.resolve())
display(pd.DataFrame({split: frame['hate_speech'].value_counts() for split, frame in frames.items()}).T)


## 4. Tokenizer dan data SFT

Kapitalisasi/emoji dipertahankan. Hanya isi user yang dipotong bila terlalu panjang.
Prompt dan padding di-mask `-100`; loss hanya pada label jawaban + EOS. Test ditokenisasi pada evaluasi akhir.

In [ ]:
MODEL_COMMIT = HfApi().model_info(MODEL_NAME).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_COMMIT)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
EOS_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
assert tokenizer.convert_ids_to_tokens(EOS_ID) == '<|im_end|>'
answers = {label: tokenizer.encode(LABEL_TO_ANSWER[label], add_special_tokens=False) + [EOS_ID]
           for label in LABELS}

def prompt_ids(content):
    return tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': content}],
        tokenize=True, return_dict=False, add_generation_prompt=True, enable_thinking=False)

def encode(row):
    content = f"Language: {LANGUAGES[LANGUAGE]}\nComment:\n{row['yt_comment']}"
    if INPUT_MODE == 'with_video_context':
        content += f"\n\nVideo title:\n{row['yt_title']}\nVideo description:\n{row['yt_description']}"
    tokens = tokenizer.encode(content, add_special_tokens=False)
    budget = MAX_LENGTH - max(map(len, answers.values()))
    available = budget - len(prompt_ids('')) - 8
    if available < 16:
        raise ValueError('Naikkan MAX_LENGTH agar instruksi dan komentar muat.')
    keep = min(len(tokens), available)
    prompt = prompt_ids(content if keep == len(tokens) else tokenizer.decode(tokens[:keep]))
    while len(prompt) > budget and keep > 8:
        keep -= 8
        prompt = prompt_ids(tokenizer.decode(tokens[:keep]))
    if len(prompt) > budget:
        raise ValueError('Prompt terlalu panjang untuk MAX_LENGTH.')
    return {'id': row.get('StereoQueerEval_id', 'new'), 'prompt': prompt,
            'gold': row.get('hate_speech'), 'truncated': keep < len(tokens)}

def example(record, label=None):
    answer = answers[record['gold'] if label is None else label]
    return {'input_ids': record['prompt'] + answer, 'labels': [-100] * len(record['prompt']) + answer}

def collate(features):
    length = math.ceil(max(len(row['input_ids']) for row in features) / 8) * 8
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for row in features:
        size = len(row['input_ids'])
        batch['input_ids'].append(row['input_ids'] + [tokenizer.pad_token_id] * (length - size))
        batch['attention_mask'].append([1] * size + [0] * (length - size))
        batch['labels'].append(row['labels'] + [-100] * (length - size))
    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

train_records = [encode(row) for row in frames['train'].to_dict('records')]
val_records = [encode(row) for row in frames['val'].to_dict('records')]
train_dataset = Dataset.from_list([example(row) for row in train_records])
print('Terpotong: train', sum(r['truncated'] for r in train_records), '| val', sum(r['truncated'] for r in val_records))
print(tokenizer.decode(train_records[0]['prompt']), '\nTARGET:', LABEL_TO_ANSWER[train_records[0]['gold']], '| Label dataset:', train_records[0]['gold'])


## 5. Muat Qwen dan pasang LoRA

Setiap run memulai dari checkpoint dasar yang sama dengan adapter baru; LoRA rank 16, alpha 32, dropout 0,05.

In [ ]:
for name in ('trainer', 'model', 'base_model'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
model, loading = Qwen3_5ForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_COMMIT, dtype=torch.bfloat16, device_map={'': 0},
    attn_implementation='sdpa', output_loading_info=True)
if loading.get('missing_keys') or loading.get('mismatched_keys'):
    raise RuntimeError(f'Bobot bahasa tidak termuat lengkap: {loading}')
model.config.use_cache = False
lora = LoraConfig(task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules='all-linear', bias='none')
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.print_trainable_parameters()


## 6. Loss, prediksi, dan Trainer

Prediksi memilih probabilitas tertinggi dari **seluruh label + EOS**, tanpa normalisasi panjang.
Gold label hanya dipakai untuk metrik. Logits dihitung pada posisi jawaban saja untuk menghemat VRAM.

In [ ]:
def completion_scores(model, batch):
    targets = batch['labels'][:, 1:]
    positions = torch.where((targets != -100).any(dim=0))[0]
    output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                   logits_to_keep=positions, use_cache=False)
    targets = targets[:, positions]
    loss = F.cross_entropy(output.logits.float().reshape(-1, output.logits.shape[-1]),
                           targets.reshape(-1), ignore_index=-100, reduction='none').view_as(targets)
    return -loss.sum(dim=1), (targets != -100).sum(dim=1), output

def predict(records):
    was_training, device, rows = model.training, next(model.parameters()).device, []
    model.eval()
    autocast = torch.autocast('cuda', dtype=torch.bfloat16) if device.type == 'cuda' else nullcontext()
    try:
        with torch.inference_mode(), autocast:
            for start in tqdm(range(0, len(records), EVAL_BATCH_SIZE), desc='Prediksi'):
                chunk = records[start:start + EVAL_BATCH_SIZE]
                batch = collate([example(record, label) for record in chunk for label in LABELS])
                batch = {key: value.to(device) for key, value in batch.items()}
                scores, counts, _ = completion_scores(model, batch)
                scores, counts = scores.view(len(chunk), 3).cpu(), counts.view(len(chunk), 3).cpu()
                if not torch.isfinite(scores).all():
                    raise FloatingPointError('Skor NaN/Inf; periksa runtime sebelum melanjutkan.')
                for i, record in enumerate(chunk):
                    row = {'StereoQueerEval_id': record['id'], 'true_label': record['gold'],
                           'predicted_label': LABELS[int(scores[i].argmax())], 'truncated': record['truncated']}
                    row.update({f'log_score_{label}': float(scores[i, j]) for j, label in enumerate(LABELS)})
                    if record['gold'] in LABELS:
                        j = LABELS.index(record['gold'])
                        row['nll'] = float(-scores[i, j] / counts[i, j])
                    rows.append(row)
    finally:
        model.train(was_training)
    return pd.DataFrame(rows)

def metrics(predictions):
    report = classification_report(predictions.true_label, predictions.predicted_label,
                                   labels=LABELS, output_dict=True, zero_division=0)
    return {'accuracy': report['accuracy'], 'macro_f1': report['macro avg']['f1-score'],
            'weighted_f1': report['weighted avg']['f1-score'], 'loss': float(predictions.nll.mean())}

class SFTTrainer(Trainer):
    loss_is_scaled_for_ga = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        scores, counts, output = completion_scores(model, inputs)
        loss = -scores.sum() / counts.sum()
        return (loss, output) if return_outputs else loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval'):
        records = self.eval_dataset if eval_dataset is None else eval_dataset
        result = {f'{metric_key_prefix}_{key}': value for key, value in metrics(predict(records)).items()}
        self.log(result)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, result)
        return result

def evaluate(records, name):
    predictions = predict(records)
    predictions.to_csv(OUT / f'{name}_predictions.csv', index=False)
    report = classification_report(predictions.true_label, predictions.predicted_label,
                                   labels=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT / f'{name}_report.csv')
    matrix = confusion_matrix(predictions.true_label, predictions.predicted_label, labels=LABELS)
    pd.DataFrame(matrix, index=LABELS, columns=LABELS).to_csv(OUT / f'{name}_confusion_matrix.csv')
    print(name, metrics(predictions))
    return metrics(predictions)


## 7. Baseline zero-shot pada validation (opsional)

Adapter dinonaktifkan untuk mengukur Qwen asli. Ini pembanding untuk SFT.

In [ ]:
results = {}
if RUN_ZERO_SHOT_BASELINE:
    with model.disable_adapter():
        results['zero_shot_val'] = evaluate(val_records, 'zero_shot_val')


## 8. Training SFT

Maksimal 5 epoch; early stopping patience 2; model terbaik dipilih dari macro-F1 validation. Test tidak masuk Trainer.

In [ ]:
args = TrainingArguments(
    output_dir=str(OUT / 'checkpoints'), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, weight_decay=0.01,
    warmup_steps=math.ceil(math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS * 0.05),
    lr_scheduler_type='cosine', optim='adamw_torch', bf16=True, max_grad_norm=1.0,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
    logging_steps=10, report_to='none', seed=SEED, data_seed=SEED,
    remove_unused_columns=False, label_names=['labels'], dataloader_num_workers=0)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_records,
                     data_collator=collate, processing_class=tokenizer,
                     callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.model_accepts_loss_kwargs = False
train_result = trainer.train()
model = trainer.model
pd.DataFrame(trainer.state.log_history).to_csv(OUT / 'training_log.csv', index=False)
print('Best validation macro-F1:', trainer.state.best_metric)


## 9. Evaluasi akhir

Test internal dari training release digunakan setelah checkpoint dipilih. Jangan mengubah prompt/konfigurasi berdasarkan hasil test.

In [ ]:
if not trainer.state.best_model_checkpoint:
    raise RuntimeError('Selesaikan training sebelum evaluasi test.')
results['sft_val'] = evaluate(val_records, 'sft_val')
test_records = [encode(row) for row in frames['test'].to_dict('records')]
results['sft_test'] = evaluate(test_records, 'sft_test')
if RUN_ZERO_SHOT_BASELINE:
    with model.disable_adapter():
        results['zero_shot_test'] = evaluate(test_records, 'zero_shot_test')
display(pd.DataFrame(results).T)
matrix = pd.read_csv(OUT / 'sft_test_confusion_matrix.csv', index_col=0)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(matrix, cmap='Blues')
ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
       xlabel='Predicted', ylabel='True', title='SFT test')
plt.setp(ax.get_xticklabels(), rotation=25, ha='right')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(matrix.iloc[i, j]), ha='center', va='center')
fig.tight_layout()
fig.savefig(OUT / 'sft_test_confusion_matrix.png', dpi=150)
plt.show()
plt.close(fig)


## 10. Simpan dan download ZIP

ZIP mencakup adapter terbaik, tokenizer, konfigurasi, metrik, prediksi, dan laporan per kelas.
Checkpoint sementara tidak dimasukkan. Download melalui panel Files Modal atau link di bawah.

In [ ]:
model.save_pretrained(OUT / 'best_adapter', safe_serialization=True, save_embedding_layers=False)
tokenizer.save_pretrained(OUT / 'best_adapter')
(OUT / PROMPT_NAME).write_bytes(PROMPT_BYTES)
config = {'model': MODEL_NAME, 'revision': MODEL_COMMIT, 'language': LANGUAGE, 'input_mode': INPUT_MODE,
          'system_prompt': SYSTEM_PROMPT, 'prompt_file': PROMPT_NAME, 'prompt_sha256': PROMPT_SHA256,
          'label_to_answer': LABEL_TO_ANSWER, 'labels': LABELS, 'max_length': MAX_LENGTH,
          'enable_thinking': False, 'prediction': 'sum log P(label + im_end), no length normalization',
          'training': args.to_dict(), 'lora': lora.to_dict(), 'csv_sha256': source_hashes,
          'versions': {'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__},
          'best_validation_macro_f1': trainer.state.best_metric, 'zero_shot_baseline': RUN_ZERO_SHOT_BASELINE,
          'train_metrics': train_result.metrics}
for name, value in [('config', config), ('metrics', results)]:
    (OUT / f'{name}.json').write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
pd.DataFrame(results).T.to_csv(OUT / 'metrics.csv')
zip_path = OUT.with_suffix('.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for file in OUT.rglob('*'):
        if file.is_file() and file.relative_to(OUT).parts[0] != 'checkpoints':
            archive.write(file, file.relative_to(OUT))
print('Download sebelum sesi berakhir:', zip_path)
display(FileLink(str(zip_path)))


### Opsional: prediksi komentar baru

Menggunakan adapter terbaik yang masih ada di GPU. Untuk run bahasa lain, upload tiga CSV dan prompt bahasanya, ubah `LANGUAGE`, dan ulangi bagian 2–10.

In [ ]:
NEW_COMMENT = 'Everyone deserves to be treated with dignity.'
row = {'yt_comment': NEW_COMMENT, 'yt_title': '', 'yt_description': ''}
display(predict([encode(row)]).drop(columns=['true_label'], errors='ignore'))
